In [0]:
path='/Volumes/itransition/default/task_4/data.zip'
extract_path='/Volumes/itransition/default/task_4/ectract'
import zipfile
import os 
import re

with zipfile.ZipFile(path, 'r') as zip_ref:
    file_name=[i for i in zip_ref.namelist() if not i.endswith('._') and not i.startswith('__') and re.search(r'\.(parquet|csv|yaml)$',i)]
    
    zip_ref.extractall(extract_path,members=file_name)
    print('bajarildi')

In [0]:
import yaml 
import glob 
from functools import reduce

for i in glob.glob(extract_path+'/data/*/*.parquet'):
    parquet_df=spark.read.parquet(i)
    parquet_df.write.mode('append').saveAsTable('itransition.default.parquet_df')



yaml_path=(glob.glob(extract_path+'/data/*/*.yaml')+glob.glob(extract_path+'/data/*/*.yml'))
dfs=[]
for i in yaml_path:

    yaml_df=yaml.safe_load(open(i,'r'))
    
    if isinstance(yaml_df,list):
        df=spark.createDataFrame(yaml_df)
    elif isinstance(yaml_df,dict):
        df=spark.createDataFrame([yaml_df])
    dfs.append(df)

if dfs:
    yaml_df=reduce(lambda x, y: x.union(y), dfs)
    yaml_df.write.mode('overwrite').saveAsTable('itransition.default.yaml_df')



spark.sql('USE itransition.default')

csv_files=glob.glob(extract_path+'/data/*/*.csv')
for i in csv_files:
    csv_df=spark.read.csv(i,header=True,inferSchema=True)
    csv_df.write.mode('append').saveAsTable('csv_df')